Deterministic / Model Based 

In [48]:
import os 
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D627BD8C30>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D627BD88A0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

Input -> PII Middleware -> Agent 

In [49]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

@tool
def customer_loopkup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

agent = create_agent(
    model=model,
    tools=[customer_loopkup],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "api_key", # Should run in the try block beacuse, exception is raised here 
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ]
)
print("Agent with PII Middleware created succesfully")

Agent with PII Middleware created succesfully


In [50]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content":"My email is john@example.com and my credit card is 5105-5125-8458-6120. Can you help me??"
    }]
})

In [51]:
# The built-in credit card detector uses the Luhn algorithm to validate candidate card numbers.

In [52]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my credit card is 5105-5125-8458-6120. Can you help me??', additional_kwargs={}, response_metadata={}, id='716dbff8-39e8-4497-a383-8dd0290aaa4c'),
  AIMessage(content='I’m sorry, but I can’t process or store personal or payment information like that.\u202fIf you let me know what you need help with (without sharing sensitive details), I’ll do my best to assist you!', additional_kwargs={'reasoning_content': 'The user is providing personal data (email, credit card) and asks for help. According to policy, we must not store or process sensitive personal data. We should respond acknowledging we cannot handle credit card info, advise them not to share such info, and ask how we can help in a non-sensitive manner. No need to call any function.'}, response_metadata={'token_usage': {'completion_tokens': 123, 'prompt_tokens': 152, 'total_tokens': 275, 'completion_time': 0.258048581, 'completion_tokens_details': {'reasoning_tokens

In [53]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "My email is test@example.com "
                "and my card is 5105-1051-0510-5100. Can you help me?"
            ),
        }
    ]
})

for message in result["messages"]:
    print(type(message).__name__, ":", message.content)

HumanMessage : My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?
AIMessage : I’m happy to help! To get you the right assistance, could you let me know what you need help with (e.g., a recent transaction, a billing question, account access, etc.)? I’ll look up your account using the email you provided—no full card details are needed.


### Checkpointer is required with Human in the Loop 

In [54]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command # It is for the approval or for Human Feedback
from langchain_core.tools import tool 

@tool
def search_web(query: str)-> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str)-> str:
    """Send an email to the recipent"""
    return f"Email sent to {to} with subjects: {subject}"

@tool 
def delete_records(table:str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

hitl_agent = create_agent(
    model = model,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,
                "delete_records": True,
                "search_web": False,
            }
        ),
    ],
    checkpointer=InMemorySaver()
)

In [55]:
config = {"configurable": {"thread_id":"session_01"}}

hitl_agent.invoke(
    {"messages":[{"role":"user", "content":"Send an email to team@company.com about the Q4 results"}]},
    config=config
)

rejected_results = hitl_agent.invoke(
    Command(resume={"decisions": [{"type":"reject", "reason":"Too risky, need DBA review"}]}),
    config=config
)

In [56]:
rejected_results

{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='f57c4429-00ae-4c11-961d-65cd6a213c4d'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants an email to team@company.com about Q4 results. Need to ask for email content? The instruction says "Send an email to team@company.com about the Q4 results". Probably we need to compose a brief email. We can ask clarification? Could assume generic: "Please find attached the Q4 results..." But we need subject and body. Could ask for more details, but maybe we can generate a generic email. Let\'s assume we can send a simple email.\n\nWe need to call send_email function with to, subject, body.\n\nLet\'s draft: Subject: Q4 Results Overview. Body: Dear Team, ... Provide summary. We\'ll send.\n\n', 'tool_calls': [{'id': 'fc_4317582e-cd77-4db7-ac3e-e931146fd752', 'function': {'arguments': '{"body":"Dear Team,\\n\\nI hope you are all doing we

### Custom Guardrails 

In [57]:
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from typing import Any 

In [58]:
class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic gurdrails : Block requests containing banned keywords.
    This runs BEFORE the agent processes anything - Zero LLM cost for blocked rquests 
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime:Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type!="human": # Message type can be AIMessage also like How can I assist you, 
            return None 
        # To ensure keyword search is done on Human message only 

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked - Keyword Detected: {keyword}")
                return {
                    "messages":[ {
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content."
                            "Please rePhase your request"
                        )
                    }],
                    "jump_to":"end"
                }
        return None 


In [59]:
filtered_agent = create_agent(
    model= model, 
    tools=[search_web],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

In [60]:
result_fil = filtered_agent.invoke({
    "messages" : [{"role":"user", "content":"How can I assist you ?"}]
})
result_fil["messages"][-1].content

'I’m here to help\u202f—\u202fjust let me know what you’re interested in or what you need assistance with, and I’ll do my best to support you!'